Smartphone Data Cleaning


### Objective

Identify, investigate, and resolve confirmed data-quality issues in the smartphone dataset.

### Data Cleaning Principles

- Preserve the original raw dataset without modification.
- Investigate missing values before deciding how to handle them.
- Correct only verified data errors and inconsistencies.
- Avoid removing valid records unless there is a clear justification.
- Document all data-cleaning decisions and changes.
- Validate the cleaned dataset to ensure data quality and consistency before saving it.


In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/smartphones.csv")

# creating a copy for working 
df_clean = df.copy()

print("Original Working Directory: ", df.shape)
print(" Working dataset shape: ", df_clean.shape)

Original Working Directory:  (980, 22)
 Working dataset shape:  (980, 22)


## 

### <b> 1. Missing-Value Investigation </b>
<b> Objective:</b> Identify which columns contain missing values and how frequently they occur.

<b> Decision rule:</b> Do not automatically fill or delete missing values. First investigate why they are missing and whether the information can be recovered.

In [4]:
df_clean.head()

,brand_name,model,price,avg_rating,5G_or_not,processor_brand,num_cores,processor_speed,battery_capacity,fast_charging_available,...,internal_memory,screen_size,refresh_rate,num_rear_cameras,os,primary_camera_rear,primary_camera_front,extended_memory_available,resolution_height,resolution_width
0,apple,Apple iPhone 11,38999,7.3,0,bionic,6.0,2.65,3110.0,0,...,64,6.1,60,2,ios,12.0,12.0,0,1792,828
1,apple,Apple iPhone 11 (128GB),46999,7.5,0,bionic,6.0,2.65,3110.0,0,...,128,6.1,60,2,ios,12.0,12.0,0,1792,828
2,apple,Apple iPhone 11 Pro Max,109900,7.7,0,bionic,6.0,2.65,3500.0,1,...,64,6.5,60,3,ios,12.0,12.0,0,2688,1242
3,apple,Apple iPhone 12,51999,7.4,1,bionic,6.0,3.10,NaN,0,...,64,6.1,60,2,ios,12.0,12.0,0,2532,1170
4,apple,Apple iPhone 12 (128GB),55999,7.5,1,bionic,6.0,3.10,NaN,0,...,128,6.1,60,2,ios,12.0,12.0,0,2532,1170


In [19]:
# so finding missing values in df_clean & which have > 0 that we are showing

missing_report = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_report = missing_report[missing_report["missing_count"]> 0]
missing_report

,column,missing_count
3,avg_rating,101
5,processor_brand,20
6,num_cores,6
7,processor_speed,42
8,battery_capacity,11
10,fast_charging,211
16,os,14
18,primary_camera_front,5


In [22]:
# missing report percentage
missing_report["missing_percentage"] = (
    missing_report["missing_count"] / len(df_clean) * 100

).round(2)

missing_report = missing_report.sort_values(
    "missing_count", ascending=False
)
missing_report

,column,missing_count,missing_percentage
10,fast_charging,211,21.53
3,avg_rating,101,10.31
7,processor_speed,42,4.29
5,processor_brand,20,2.04
16,os,14,1.43
8,battery_capacity,11,1.12
6,num_cores,6,0.61
18,primary_camera_front,5,0.51


### <b> 2. Investigating Missing-Value Patterns </b>

<b>Objective:</b> Determine whether missing values in one column are associated with values in another column.

<b>Why:</b> Missing data may reflect unavailable specifications, incomplete records, or a relationship between features. We should investigate before choosing a cleaning strategy.

2.1 Check missing fast_charging values

In [32]:
charging_missing = df_clean["fast_charging"].isna()
print("Missing fast_charging values: ")
print(charging_missing.sum())

print("fast_charging_available distribution:")
print(df_clean["fast_charging_available"].value_counts(dropna=False))

print("\nCharging availabilty for missing fast_charging:")
print(
    df_clean.loc[
        charging_missing, "fast_charging_available"
    ].value_counts(dropna=False)
)

Missing fast_charging values: 
211
fast_charging_available distribution:
fast_charging_available
1    837
0    143
Name: count, dtype: int64

Charging availabilty for missing fast_charging:
fast_charging_available
0    143
1     68
Name: count, dtype: int64


2.2 Check missing ratings by brand